In [ ]:
import pandas as pd

data = pd.read_json("data/preprocessed_data.jsonl", lines=True)
data.head(10)

,src,tgt
0,وي انا تعجبني برشا,oui ene t3jbni barcha
1,تبارك اله عليه,tbarek allah 3lih
2,امرأة أنيقة و سلسة في خطابها و ذكية و متواضعة,imra2a ani9a w salisa fi 5itabiha w dhakiya w ...
3,جعفر يحب بلادو ويعمل في الخير يعاون في الناس و...,ja3far y7eb bladou w ya3mel fil 5ir y3awen fi ...
4,محلاها,mahleha
5,عماد دغيج برافو واله راجل موش كيما بعض السياسي...,3imed d8ij bravo wlh rajel mouch kima ba3th es...
6,ليلوشا محلاك تهبل تقتل تقتل 3,laylouchaa mahlek t-habel to99tel to9tel 3
7,مرا تونسية بأتم معنى الكلمة,mra tounsia b2tam ma3na lklima
8,معلم,ma3alme
9,تعجبني برشا,te3jebni barcha


In [2]:
#Tokenization : for translation tasks, we need to tokenize the text into character level tokens.
#because if we do it at word level, Barcha , baarcha,barch2 will be considered as different words,
# but if we do it at character level, they will be considered as the same word.

#first we build the vocab
#<pad> : padding token : used to pad sequences to the same length in a batch.
#<sos> : start of sequence token : used to indicate the start of a sequence
#<eos> : end of sequence token : used to indicate the end of a sequence
#<unk> : unknown token : used to represent tokens that are not in the vocabulary

arabic_vocab  = ["<pad>", "<sos>", "<eos>", "<unk>"]
arabizi_vocab = ["<pad>", "<sos>", "<eos>", "<unk>"]

# FIX: loop over each side separately (zip(src, tgt) stopped at the shorter string and skipped characters)
for char in sorted(set("".join(data["src"]))):
    arabic_vocab.append(char)
for char in sorted(set("".join(data["tgt"]))):
    arabizi_vocab.append(char)

arabic_vocab_size  = len(arabic_vocab)
arabizi_vocab_size = len(arabizi_vocab)

print(f"Arabic vocab size: {arabic_vocab_size}")
print(f"Arabic vocab: {arabic_vocab}")
print(f"Arabizi vocab size: {arabizi_vocab_size}")
print(f"Arabizi vocab: {arabizi_vocab}")


Arabic vocab size: 64
Arabic vocab: ['<pad>', '<sos>', '<eos>', '<unk>', ' ', '"', '%', "'", '(', ')', '*', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', '_', 'ء', 'آ', 'أ', 'ؤ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ى', 'ي', 'ڤ', 'ڨ']
Arabizi vocab size: 52
Arabizi vocab: ['<pad>', '<sos>', '<eos>', '<unk>', ' ', '"', '%', "'", '(', ')', '*', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [3]:
#funciton to convert tokens to indices and vice versa
arabic_token2idx = {token:i for i, token in enumerate(arabic_vocab)}
arabic_idx2token = {i:token for i, token in enumerate(arabic_vocab)}
arabizi_token2idx = {token:i for i, token in enumerate(arabizi_vocab)}
arabizi_idx2token = {i:token for i, token in enumerate(arabizi_vocab)}

In [4]:
def sentence_to_one_hot_vector(sentence,token2idx,vocab):
    one_hot = []
    for ch in sentence:
        if not(ch in vocab) :
            one_hot.append(token2idx["<unk>"])
        else:
            one_hot.append(token2idx[ch])

    return one_hot

In [6]:
# let_s build the embeddings of the araboc part :

arabic_corpus =""
for row in data.iterrows():
    arabic_corpus += row[1]["src"] + " "

In [ ]:
sentence_to_one_hot_vector('هاي')

33

In [ ]:
#builing the encoder 

class Encoder(nn.Module):
    def __init__(self,vocab_size,emb_size,hidden_size,num_layers,dropout):
        super().__init__()

        self.embeddings = nn.Embedding(vocab_size,emb_size,padding_idx=stoi['<pad>'])
        self.dropout = nn.Dropout(p=dropout)

        self.lstm = nn.LSTM(input_size=emb_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers>1 else 0)               #(B, T, emb_dim) → outputs (B, T, H) + (h_n, c_n) each (L, B, H)



    def forward(self,src) : 
        emb = self.embeddings(src)
        embedded = self.dropout(emb)
        outputs , (h_n , c_n) = self.lstm(embedded)

        return outputs , (h_n , c_n)

In [ ]:
#building the decoder 
class Decoder(nn.Module):
    def __init__(self,vocab_size,emb_size,hidden_size,num_layers,dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size,emb_size,padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(input_size=emb_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers>1 else 0) 

        self.linear = nn.Linear(hidden_size,vocab_size)

    def forward(self,input,hidden,cell):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        output, (h,c) = self.lstm(embedded,(hidden,cell))
        logits = self.linear(output.squeeze(1))
        return logits , (h,c)


In [ ]:

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.enc = encoder
        self.dec = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):         
        B, T = tgt.shape
        V = self.dec.linear.out_features                        

        outputs = torch.zeros(B, T, V, device=src.device)          

        _, (h, c) = self.enc(src)                                   
        input = tgt[:, 0]                                   

        for t in range(1, T):                                        # FIX 3: start at 1
            logits, (h, c) = self.dec(input, h, c)                   # FIX 2: self.dec, not dec
            outputs[:, t] = logits                                   # prediction that should equal tgt[:, t]

            use_teacher = random.random() < teacher_forcing_ratio
            input = tgt[:, t] if use_teacher else logits.argmax(1)   # FIX 3: tgt[:, t], not t+1

        return outputs

In [ ]:
VOCAB_SIZE = len(itos)
model = Seq2Seq(Encoder(VOCAB_SIZE, 32, 128, 1, 0.1),
                Decoder(VOCAB_SIZE, 32, 128, 1, 0.1))

optimizer = torch.optim.Adam(model.parameters() , lr =18e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)         #CrossEntropyLoss that ignores padding (ID 0)
N_STEPS = 30000

# ---- training loop ----
for step in range(1, N_STEPS + 1):
    model.train()                                   # dropout ON

    src, tgt = make_batch(64)

    optimizer.zero_grad()                           # 1. clear old gradients
    outputs = model(src,tgt,0.5)

    V = outputs.shape[-1]
    pred    = outputs[:,1:].reshape(-1,V)
    answers = tgt[:,1:].reshape(-1)
    loss = criterion(pred, answers)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   #clip gradients to max norm 1.0
    optimizer.step()

    if step % 200 == 0:
        print(f"step {step:4d} | loss {loss.item():.3f}")

In [ ]:
@torch.no_grad()
def translate(model, s, max_len=50):
    model.eval()

    src = torch.tensor([encode(s)])     #encode s (no <sos>/<eos>), make it a tensor of shape (1, len(s))
    _, (h, c) =model.enc(src)       #run the encoder

    input = torch.tensor([stoi['<sos>']])   # tensor of shape (1,) containing the <sos> ID
    result = []
    for _ in range(max_len):
        logits, (h, c) = model.dec(input,h,c) 
        next_id = logits.argmax(1).item()
        if next_id == stoi['<eos>']:
            break
        result.append(itos[next_id])
        input = torch.tensor([next_id]) # next_id as a tensor of shape (1,)

    return "".join(result)

In [ ]:
def evaluate(model, pairs, show_errors=5):
    correct = 0
    errors = []
    for src, expected in pairs:
        pred = translate(model, src)
        if pred == expected:
            correct += 1
        elif len(errors) < show_errors:
            errors.append((src, expected, pred))

    acc = correct / len(pairs)
    print(f"exact-match accuracy: {acc:.1%}  ({correct}/{len(pairs)})")
    for src, expected, pred in errors:
        print(f"  {src:14s} expected {expected:14s} got {pred}")
    return acc
